# 4–8-qubit TFIM local certification scaling benchmark

This notebook evaluates the manuscript's local metric-uncertainty certificate on open transverse-longitudinal Ising chains with `J=1`, `hx=0.8`, and `hz=0.3` for 4–8 qubits. Because the theorem is local at an exactly represented ground-state optimum, the benchmark uses an exact-ground-state-centered local unitary chart generated by single-qubit `Y_i` and nearest-neighbor `Z_i Z_{i+1}` directions. It is a local scaling diagnostic, not a claim about state-preparation cost or global VQE convergence.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
I=np.eye(2,dtype=complex)
X=np.array([[0,1],[1,0]],complex)
Y=np.array([[0,-1j],[1j,0]],complex)
Z=np.array([[1,0],[0,-1]],complex)
def kron_all(ops):
    out=np.array([[1.0]],complex)
    for op in ops: out=np.kron(out,op)
    return out
def pauli_string(n,mapping):
    return kron_all([mapping.get(i,I) for i in range(n)])
def tfim_hamiltonian(n,J=1.0,hx=0.8,hz=0.3):
    H=np.zeros((2**n,2**n),complex)
    for i in range(n-1): H-=J*pauli_string(n,{i:Z,i+1:Z})
    for i in range(n):
        H-=hx*pauli_string(n,{i:X})
        H-=hz*pauli_string(n,{i:Z})
    return H

In [ ]:
def local_tfim_certificate(n,J=1.0,hx=0.8,hz=0.3):
    H=tfim_hamiltonian(n,J,hx,hz)
    evals,evecs=np.linalg.eigh(H)
    psi0=evecs[:,0]; E0=float(evals[0]); eye=np.eye(2**n,dtype=complex)
    generators=[pauli_string(n,{i:Y}) for i in range(n)] + [pauli_string(n,{i:Z,i+1:Z}) for i in range(n-1)]
    dpsi=[]
    for P in generators:
        mean=float(np.real(np.vdot(psi0,P@psi0)))
        dpsi.append((-0.5j)*(P-mean*eye)@psi0)
    p=len(dpsi); A=H-E0*eye
    G=np.array([[np.real(np.vdot(di,dj)) for dj in dpsi] for di in dpsi])
    Hess=np.array([[2*np.real(np.vdot(di,A@dj)) for dj in dpsi] for di in dpsi])
    ew,U=np.linalg.eigh(G); Uact=U[:,ew>1e-10]
    Gact=Uact.T@G@Uact; Hact=Uact.T@Hess@Uact
    gvals,Q=np.linalg.eigh(Gact); Gmhalf=Q@np.diag(1/np.sqrt(gvals))@Q.T
    S0=Gmhalf@Hact@Gmhalf
    hvals=np.linalg.eigvalsh(Hact); svals=np.linalg.eigvalsh(S0)
    gap=float(evals[1]-evals[0]); width=float(evals[-1]-evals[0])
    ratio=width/gap; kH=float(hvals[-1]/hvals[0]); kS=float(svals[-1]/svals[0])
    gmin=float(gvals[0]); gmax=float(gvals[-1])
    epscrit=0.5*gmin*((gap/width)*kH-1.0)
    epshalf=0.5*epscrit; Khalf=ratio*(1+2*epshalf/gmin)
    return dict(n_qubits=n,p_parameters=p,active_rank=len(gvals),gap=gap,width=width,W_over_Delta=ratio,g_min=gmin,g_max=gmax,kappa_H=kH,kappa_S0=kS,epsilon_crit=epscrit,epsilon_halfcrit=epshalf,K_at_halfcrit=Khalf,commutator_norm=float(np.linalg.norm(Gact@Hact-Hact@Gact,2)))

In [ ]:
df=pd.DataFrame([local_tfim_certificate(n) for n in range(4,9)])
df.round(8)

In [ ]:
assert np.all(df['epsilon_crit']>0)
assert np.all(df['K_at_halfcrit']<df['kappa_H'])
assert np.all(df['kappa_S0']<=df['W_over_Delta']+1e-10)
print('All scaling-certificate checks passed.')